# Frozen AfroXLMR vs the deployed three-head model

Compare two setups on one labeled file. Nothing in this notebook invents a metric: a row stays `n/a` until that system has actually been scored.

| System | What is trained |
| --- | --- |
| Frozen encoder + cloze prompt (zero training of heads) | Nothing. `Davlan/afro-xlmr-large` fill-mask, and only when every class surface form is a single vocab token. |
| Frozen encoder + nearest centroid | Nothing inside the encoder. Class means come from a disjoint N-shot set. |
| Frozen encoder + linear probe | Nothing inside the encoder. One linear head per task, fit on those N-shot rows only. |
| Fine-tuned shared encoder + 3 heads | Already trained. This notebook does not train it. It scores the deployed checkpoint through `POST /predict_batch`, the same call as `eval/run.mjs`. |

`Davlan/afro-xlmr-large` is an encoder-only XLM-RoBERTa adapted with masked language modeling (Alabi et al. 2022). It is not a chat model, so "ask it to generate the label" is not a fair test. Fill-mask is the prompt that matches the checkpoint. Issue names such as `Airtel_Money_Reversal` are several words. If they are not one token each, cloze stops and says so. It does not substitute a hand-picked verbalizer.

This repository does not name the backbone inside `joint_big_model.pt`. Baseline B is the live three-head service: 10 issue classes, binary sentiment, urgency decoded to `low` / `medium` / `emergency`. The training setup calls the urgency head CORN. That class is not in this repo, and a local `.pt` is not unpickled.

## Required files

- `notebooks/requirements.txt` when you turn a model flag on (`torch`, `transformers`).
- Gold CSV or xlsx with `text`, `issue`, `sentiment`, `urgency`, and optional `id`. Same columns as `eval/README.md`. Use the file you already pass to `node eval/run.mjs --csv`.
- Optional few-shot file that does not repeat gold ids or exact message text. Otherwise set `split_n_shot_from_gold` and read the banner: the table is then the remainder, not the full gold file.
- Network for Modal (`call_live_api`) or Hugging Face (`run_frozen_encoder`, cloze flags). The first Modal request can take 40–120 seconds.
- Checkpoint path the live health payload reports: `/data/ckpt/joint_big_model.pt`. Set `local_checkpoint` if a copy is on disk. Detection only.

`allow_smoke_fixture` loads `eval/synthetic-heldout.smoke.csv` so you can see the wiring. That file is not gold and those numbers are not model quality.

Do not commit the gold file, the few-shot file, `.pt` weights, or an executed copy of this notebook that still shows customer text.


In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "notebooks" / "metrics.py").is_file() else HERE.parent
if not (ROOT / "notebooks" / "metrics.py").is_file():
    raise SystemExit(f"Run from the repo root or from notebooks/. cwd={HERE}")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

GOLD_ROWS = None
SCORED_ROWS = None
TRAIN_ROWS = None
SPLIT_NOTES = []
GOLD_META = {"smoke": False, "path": "", "warning": ""}
BASELINE_B = None
BASELINE_B_NOTE = "call_live_api is false. No Baseline B metric is computed."
CLOZE = {"issue": None, "sentiment": None, "urgency": None}
CLOZE_NOTES = {
    "issue": "run_cloze_diagnostic is false. No cloze metric.",
    "sentiment": "run_cloze_diagnostic is false. No cloze metric.",
    "urgency": "run_cloze_diagnostic is false. No cloze metric.",
}
FROZEN_CENTROID = None
FROZEN_LINEAR = None
CENTROID_NOTE = "run_frozen_encoder is false. No frozen-encoder metric is computed."
LINEAR_NOTE = CENTROID_NOTE
print(f"repo root: {ROOT}")


## CONFIG

Edit this cell, then run the notebook from the top. Flags that call Modal or download the encoder default to false.


In [ ]:
CONFIG = {
    # Local gold/val export. Same columns as eval/run.mjs. Do not commit this file.
    "gold_csv": "",
    "gold_xlsx": "",
    # Disjoint N-shot file for the frozen probe. Must not overlap gold ids or exact texts.
    "fewshot_csv": "",
    "fewshot_xlsx": "",
    "xlsx_sheet": 0,
    # Wiring check only. Prints a smoke banner and is not a model-quality result.
    "allow_smoke_fixture": False,
    # Opt in to carve N per issue class out of gold. The table then uses the remainder for every system.
    "split_n_shot_from_gold": False,
    "n_per_issue_class": 4,
    "split_seed": 0,
    # Baseline B. The deployed three-head service. Cold start can take 40–120s.
    "call_live_api": False,
    "api_url": "",
    "chunk_size": 20,
    # Detected, not torch.loaded. Health reports /data/ckpt/joint_big_model.pt on Modal.
    "local_checkpoint": "",
    "encoder_name": "Davlan/afro-xlmr-large",
    "run_cloze_diagnostic": False,
    "run_cloze_score": False,
    "run_frozen_encoder": False,
    "max_length": 128,
    "cloze_max_length": 192,
    "embed_batch_size": 8,
    "device": "auto",
    # Empty means "the label string itself must be one vocab token". Do not fill this in to force a number.
    "verbalizers": {"issue": {}, "sentiment": {}, "urgency": {}},
    "cloze_templates": {
        "issue": "Customer message: {text} Issue: <mask>.",
        "sentiment": "Customer message: {text} Sentiment: <mask>.",
        "urgency": "Customer message: {text} Urgency: <mask>.",
    },
}
print("encoder:", CONFIG["encoder_name"])
print("call_live_api:", CONFIG["call_live_api"])
print("run_frozen_encoder:", CONFIG["run_frozen_encoder"])
print("run_cloze_diagnostic:", CONFIG["run_cloze_diagnostic"], "run_cloze_score:", CONFIG["run_cloze_score"])


## Labeled rows

Gold stays the file you configured. The frozen probe is fit on a different set: the few-shot file, or the opt-in holdout. If those rows overlap gold by id or exact text, the probe is not fit.


In [ ]:
from notebooks.gold import GoldLoadError, find_leakage, resolve_labeled, support_counts
from notebooks.probe import holdout_n_shot

GOLD_ROWS = None
SCORED_ROWS = None
TRAIN_ROWS = None
SPLIT_NOTES = []
GOLD_META = {"smoke": False, "path": "", "warning": ""}

try:
    gold = resolve_labeled(CONFIG, ROOT, "gold")
except GoldLoadError as err:
    gold = {"rows": None, "reason": str(err), "smoke": False, "path": "", "warning": ""}
    print(err)

if gold.get("rows") is None:
    print(gold.get("reason") or "No gold rows.")
else:
    GOLD_ROWS = gold["rows"]
    GOLD_META = {"smoke": gold["smoke"], "path": gold["path"], "warning": gold.get("warning") or ""}
    if GOLD_META["warning"]:
        print(GOLD_META["warning"])
    print(f"loaded {len(GOLD_ROWS)} gold rows from {GOLD_META['path']} ({len(gold['skipped'])} skipped)")
    try:
        fewshot = resolve_labeled(CONFIG, ROOT, "fewshot")
    except GoldLoadError as err:
        fewshot = {"rows": None, "reason": str(err)}
        print(err)
    if fewshot.get("rows") is not None:
        if fewshot.get("warning"):
            print(fewshot["warning"])
        problems = find_leakage(fewshot["rows"], GOLD_ROWS)
        if problems:
            print("Few-shot file overlaps gold. The probe will not be fit.")
            for problem in problems:
                print(" ", problem)
            SPLIT_NOTES = problems
            SCORED_ROWS = GOLD_ROWS
        else:
            TRAIN_ROWS = fewshot["rows"]
            SCORED_ROWS = GOLD_ROWS
            SPLIT_NOTES = [
                f"few-shot file {fewshot['path']} ({len(TRAIN_ROWS)} rows); scored set is the full gold file"
            ]
    elif CONFIG["split_n_shot_from_gold"]:
        TRAIN_ROWS, SCORED_ROWS, SPLIT_NOTES = holdout_n_shot(
            GOLD_ROWS, int(CONFIG["n_per_issue_class"]), int(CONFIG["split_seed"])
        )
        print("Opt-in split: every system below is scored on the remainder, not the full gold file.")
        if not TRAIN_ROWS:
            print("No class had more than n rows, so nothing was held out to fit.")
    elif CONFIG.get("fewshot_csv") or CONFIG.get("fewshot_xlsx"):
        SCORED_ROWS = GOLD_ROWS
        SPLIT_NOTES = [
            (fewshot.get("reason") or "Few-shot file did not load.")
            + " Frozen probes will not be fit."
        ]
    else:
        SCORED_ROWS = GOLD_ROWS
        SPLIT_NOTES = [
            "No few-shot file and split_n_shot_from_gold is false. Frozen probes will not be fit."
        ]
    for note in SPLIT_NOTES:
        print(note)
    print(f"scored rows: {len(SCORED_ROWS)}")
    if TRAIN_ROWS:
        print(f"fit rows: {len(TRAIN_ROWS)}")
        print("fit support:", support_counts(TRAIN_ROWS))


## Baseline B — fine-tuned shared encoder + 3 heads

`checkpoint_status` looks for `local_checkpoint`, then `MULTIHEAD_CKPT`, then `/data/ckpt/joint_big_model.pt`. A file that exists is listed and not loaded.

Set `call_live_api` to score the deployment. API choice matches `eval/run.mjs`: `api_url`, then `MULTIHEAD_API_URL`, then `NEXT_PUBLIC_API_URL`, then the known Modal host. Predictions are requested only for the scored rows, in that order. Message text is not printed.


In [ ]:
import json

from notebooks.baseline_b import BaselineBError, checkpoint_status, fetch_health, predict_batch, resolve_api_base

print(json.dumps(checkpoint_status(CONFIG["local_checkpoint"]), indent=2))
BASELINE_B = None
BASELINE_B_NOTE = "call_live_api is false. No Baseline B metric is computed."
if SCORED_ROWS is None:
    BASELINE_B_NOTE = "No scored rows, so Baseline B was not called."
    print(BASELINE_B_NOTE)
elif CONFIG["call_live_api"]:
    api = resolve_api_base(CONFIG["api_url"])
    print(f"api: {api}")
    print("The first request can take 40–120s while Modal loads the checkpoint.")
    try:
        health = fetch_health(api)
        print(json.dumps(health, indent=2))
        BASELINE_B = predict_batch(
            api,
            [row["text"] for row in SCORED_ROWS],
            chunk_size=int(CONFIG["chunk_size"]),
        )
        reported = health.get("checkpoint", "unreported")
        BASELINE_B_NOTE = (
            "fine-tuned shared encoder + 3 heads, scored from POST /predict_batch "
            f"(health checkpoint {reported}). Urgency is the decoded label. CORN is not reimplemented here."
        )
    except BaselineBError as err:
        BASELINE_B = None
        BASELINE_B_NOTE = f"Baseline B was not scored: {err}"
        print(BASELINE_B_NOTE)
else:
    print(BASELINE_B_NOTE)


## Cloze prompt on the masked LM

This cell loads the tokenizer when `run_cloze_diagnostic` is true and prints how each class encodes. `run_cloze_score` also loads `AutoModelForMaskedLM` and scores a head only when `cloze_readiness` says every label is one non-unk token with no collisions. Other heads stay blank in the table.


In [ ]:
CLOZE = {"issue": None, "sentiment": None, "urgency": None}
CLOZE_NOTES = {
    head: "run_cloze_diagnostic is false. No cloze metric."
    for head in ("issue", "sentiment", "urgency")
}
if not CONFIG["run_cloze_diagnostic"]:
    print(CLOZE_NOTES["issue"])
elif SCORED_ROWS is None:
    for head in CLOZE_NOTES:
        CLOZE_NOTES[head] = "No scored rows, so cloze was not run."
    print(CLOZE_NOTES["issue"])
else:
    try:
        from transformers import AutoTokenizer
    except ImportError as err:
        message = f"transformers is not installed ({err}). pip install -r notebooks/requirements.txt"
        for head in CLOZE_NOTES:
            CLOZE_NOTES[head] = message
        print(message)
    else:
        from notebooks.cloze import ClozeError, cloze_readiness, labels_for_head, load_masked_lm, predict_cloze
        from notebooks.probe import resolve_device

        tokenizer = AutoTokenizer.from_pretrained(CONFIG["encoder_name"])
        print(f"mask token: {tokenizer.mask_token!r}  unk: {tokenizer.unk_token!r}")
        verbalizers = CONFIG.get("verbalizers") or {}
        readiness = {}
        for head in ("issue", "sentiment", "urgency"):
            labels = labels_for_head(SCORED_ROWS, head)
            readiness[head] = cloze_readiness(tokenizer, labels, verbalizers.get(head) or {})
            print(f"\n{head} ready={readiness[head]['ready']}")
            print(readiness[head]["reason"])
            for label, info in readiness[head]["mapping"].items():
                print(f"  {label}: surface={info['surface']!r} tokens={info['tokens']}")
        if not CONFIG["run_cloze_score"]:
            for head in CLOZE_NOTES:
                if readiness[head]["ready"]:
                    CLOZE_NOTES[head] = "single-token cloze is possible. Set run_cloze_score true to score it. No metric yet."
                else:
                    CLOZE_NOTES[head] = "cloze not scored: " + readiness[head]["reason"]
        else:
            try:
                device = resolve_device(CONFIG["device"])
                print(f"cloze device: {device}")
                model, tokenizer, _torch = load_masked_lm(CONFIG["encoder_name"], device)
            except (ClozeError, OSError, ImportError) as err:
                for head in CLOZE_NOTES:
                    CLOZE_NOTES[head] = f"cloze model was not loaded: {err}"
                print(CLOZE_NOTES["issue"])
            else:
                texts = [row["text"] for row in SCORED_ROWS]
                for head in ("issue", "sentiment", "urgency"):
                    if not readiness[head]["ready"]:
                        CLOZE_NOTES[head] = "cloze not scored: " + readiness[head]["reason"]
                        continue
                    try:
                        CLOZE[head] = predict_cloze(
                            model,
                            tokenizer,
                            texts,
                            readiness[head]["mapping"],
                            CONFIG["cloze_templates"][head],
                            device,
                            int(CONFIG["embed_batch_size"]),
                            int(CONFIG["cloze_max_length"]),
                        )
                    except ClozeError as err:
                        CLOZE[head] = None
                        CLOZE_NOTES[head] = f"cloze not scored: {err}"
                    else:
                        CLOZE_NOTES[head] = (
                            "frozen encoder + cloze prompt (zero training of heads); "
                            "single-token verbalizer only"
                        )
        for head, note in CLOZE_NOTES.items():
            print(f"{head}: {note}")


## Frozen encoder, N-shot head only

Mean-pool the last hidden state with the attention mask, L2-normalize, and fit either a nearest centroid or a linear probe. `requires_grad` is false and the forward pass is under `torch.no_grad`. The encoder is not multi-task fine-tuned. The urgency probe is a nominal classifier, not a CORN head.

Both probes share one embedding of the fit rows and the scored rows, so they are not two different encoders.


In [ ]:
FROZEN_CENTROID = None
FROZEN_LINEAR = None
CENTROID_NOTE = "run_frozen_encoder is false. No frozen-encoder metric is computed."
LINEAR_NOTE = CENTROID_NOTE
if not CONFIG["run_frozen_encoder"]:
    print(CENTROID_NOTE)
elif SCORED_ROWS is None:
    CENTROID_NOTE = "No scored rows, so the frozen encoder was not run."
    LINEAR_NOTE = CENTROID_NOTE
    print(CENTROID_NOTE)
elif not TRAIN_ROWS:
    CENTROID_NOTE = "No disjoint N-shot rows, so the frozen encoder was not fit. " + " ".join(SPLIT_NOTES)
    LINEAR_NOTE = CENTROID_NOTE
    print(CENTROID_NOTE)
else:
    from notebooks.probe import (
        embed_texts,
        load_encoder,
        predict_centroid_heads,
        predict_linear_heads,
        resolve_device,
    )

    device = resolve_device(CONFIG["device"])
    print(f"device: {device}")
    print(f"Loading {CONFIG['encoder_name']}. The first download is the full encoder checkpoint.")
    model, tokenizer, _torch = load_encoder(CONFIG["encoder_name"], device)
    trainable = sum(int(param.requires_grad) for param in model.parameters())
    print(f"trainable tensors: {trainable}")
    if trainable != 0:
        raise RuntimeError("encoder was not frozen")
    texts = [row["text"] for row in TRAIN_ROWS] + [row["text"] for row in SCORED_ROWS]
    vectors = embed_texts(
        model,
        tokenizer,
        texts,
        device,
        int(CONFIG["embed_batch_size"]),
        int(CONFIG["max_length"]),
    )
    n_train = len(TRAIN_ROWS)
    train_x = vectors[:n_train]
    test_x = vectors[n_train:]
    FROZEN_CENTROID = predict_centroid_heads(TRAIN_ROWS, train_x, test_x)
    FROZEN_LINEAR = predict_linear_heads(TRAIN_ROWS, train_x, test_x)
    CENTROID_NOTE = (
        f"frozen encoder + nearest centroid. {CONFIG['encoder_name']} weights not updated. "
        f"Fit on {n_train} N-shot rows only."
    )
    LINEAR_NOTE = (
        f"frozen encoder + linear probe. {CONFIG['encoder_name']} weights not updated. "
        f"Tiny head fit on {n_train} N-shot rows only. Nominal urgency labels, not CORN."
    )
    print(CENTROID_NOTE)
    print(LINEAR_NOTE)


## Same-set metrics

Issue macro-F1 and sentiment macro-F1 match `eval/run.mjs`: the unweighted mean of per-class F1 over gold labels with support. Sentiment F1 (negative) is only the `negative` class. Urgency κ is quadratic-weighted Cohen's kappa on `{low, medium, emergency}`, and emergency recall is gold emergency predicted emergency. False-emergency rate is the harness's paired rate.

A cloze head that was not scored stays `n/a`. It is not filled with a dummy label.


In [ ]:
import json

from notebooks.metrics import format_comparison, metric_row, partial_triage, triage_metrics

records = []
if BASELINE_B is None or SCORED_ROWS is None:
    records.append(metric_row("fine-tuned shared encoder + 3 heads", BASELINE_B_NOTE, None))
else:
    records.append(
        metric_row(
            "fine-tuned shared encoder + 3 heads",
            BASELINE_B_NOTE,
            triage_metrics(SCORED_ROWS, BASELINE_B),
        )
    )

cloze_note = " | ".join(f"{head}: {CLOZE_NOTES[head]}" for head in ("issue", "sentiment", "urgency"))
if SCORED_ROWS is not None and any(value is not None for value in CLOZE.values()):
    cloze_metrics = partial_triage(SCORED_ROWS, CLOZE["issue"], CLOZE["sentiment"], CLOZE["urgency"])
else:
    cloze_metrics = None
records.append(metric_row("frozen encoder + cloze prompt (zero training of heads)", cloze_note, cloze_metrics))

if FROZEN_CENTROID is None or SCORED_ROWS is None:
    records.append(metric_row("frozen encoder + nearest centroid (N-shot, encoder not updated)", CENTROID_NOTE, None))
else:
    records.append(
        metric_row(
            "frozen encoder + nearest centroid (N-shot, encoder not updated)",
            CENTROID_NOTE,
            triage_metrics(SCORED_ROWS, FROZEN_CENTROID),
        )
    )

if FROZEN_LINEAR is None or SCORED_ROWS is None:
    records.append(metric_row("frozen encoder + linear probe (N-shot, encoder not updated)", LINEAR_NOTE, None))
else:
    records.append(
        metric_row(
            "frozen encoder + linear probe (N-shot, encoder not updated)",
            LINEAR_NOTE,
            triage_metrics(SCORED_ROWS, FROZEN_LINEAR),
        )
    )

print(format_comparison(records))
if GOLD_META.get("smoke"):
    print("\nThese rows used the synthetic smoke fixture. Do not quote them as model quality.")

scored_ids = [] if SCORED_ROWS is None else [row["id"] for row in SCORED_ROWS]
gold_ids = [] if GOLD_ROWS is None else [row["id"] for row in GOLD_ROWS]
out_dir = ROOT / "notebooks" / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)
report_path = out_dir / "baseline_comparison.json"
payload = {
    "smoke": bool(GOLD_META.get("smoke")),
    "gold_path": GOLD_META.get("path") or None,
    "n_gold": None if GOLD_ROWS is None else len(GOLD_ROWS),
    "n_scored": None if SCORED_ROWS is None else len(SCORED_ROWS),
    "scored_full_gold_file": bool(scored_ids) and scored_ids == gold_ids,
    "include_text": False,
    "systems": records,
}
report_path.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
print(f"wrote {report_path} (metrics and notes only, no message text)")
